# Seed Phrase Recoverer — Colab / Kaggle

Recovers the words you forgot from a **partial BIP39 seed phrase**, by deriving
addresses for EVM, Bitcoin, Dogecoin, Ripple XRP, Tron, Solana and Cardano and
matching them against **addresses you already know**.

**You must supply at least one target address** (`network:address`). The tool never
scans the blockchain and never looks for "any wallet with a balance" — it only reports
a phrase if it reproduces *your* address.

Runtime note: this is CPU code. A Colab CPU runtime has 2 cores; a Kaggle or vast.ai
instance with more cores scales almost linearly. See `spr bench`.

## 1. Get the project and install the toolchain

Either:
* set `PROJECT_URL` below to a git URL of this repository, or
* run the cell and upload `spr.zip` when asked.

In [ ]:
PROJECT_URL = ""   # e.g. "https://github.com/you/seed-phrase-recoverer.git"
PROJECT_DIR = "seed-phrase-recoverer"

import os, subprocess, sys

def sh(cmd, **kw):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=True, **kw)

if os.path.isdir(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
elif PROJECT_URL:
    sh(f"git clone --depth 1 {PROJECT_URL} {PROJECT_DIR}")
    os.chdir(PROJECT_DIR)
else:
    try:
        from google.colab import files
        print("Upload spr.zip (the repository archive)")
        up = files.upload()
        name = next(iter(up))
        sh(f"unzip -o -q {name} -d _extracted")
        root = os.listdir("_extracted")[0]
        os.chdir(os.path.join("_extracted", root))
    except Exception as exc:
        raise SystemExit(f"Set PROJECT_URL or upload the archive (error: {exc})")

print("project:", os.getcwd())
print(os.listdir()[:12])

## 2. Build and verify (a few minutes the first time)

`scripts/setup.sh` installs Rust if needed, builds the release binary, then runs the
18 published derivation vectors and a throughput benchmark.

In [ ]:
!bash scripts/setup.sh

## 3. Configure your search

* `PHRASE` — what you remember, `?` for each forgotten word (12/15/18/21/24 words total)
* `TARGETS` — one `network:address` per line; any network works, several are allowed
* Optional: `RESTRICT` (e.g. `"3=lucky,lonely"`) when you know a position is only one
  of a few words, `THREADS`, `SHARD` (for splitting over several runtimes)

In [ ]:
PHRASE = "abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon ?"
TARGETS = "evm:0x9858EfFD232B4033E47d90003D41EC34EcaEda94"
NETWORKS = ""          # empty = networks taken from the targets, e.g. "evm,btc,solana"
RESTRICT = ""          # e.g. "3=lucky,lonely"
MATCH = "exact"        # exact | prefix | suffix | contains
PASSPHRASE = ""        # BIP39 "25th word" if you used one
BREADTH = 1            # addresses per path
THREADS = 0            # 0 = all cores
SHARD = ""             # e.g. "1/4" on the first of four runtimes

## 4. Inspect the plan (no CPU spent)

In [ ]:
import subprocess
cmd = ["target/release/spr", "recover", "--dry-run", "--phrase", PHRASE,
       "--target", TARGETS.splitlines()[0].strip()]
if NETWORKS: cmd += ["--network", NETWORKS]
if PASSPHRASE: cmd += ["--passphrase", PASSPHRASE]
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

## 5. Run it

In [ ]:
import os, subprocess
os.makedirs("targets", exist_ok=True)
open("targets/targets.txt", "w").write(TARGETS)

cmd = ["target/release/spr", "recover", "--phrase", PHRASE,
       "--target-file", "targets/targets.txt", "--match", MATCH,
       "--breadth", str(BREADTH), "--threads", str(THREADS),
       "--json", "targets/results.json"]
if NETWORKS: cmd += ["--network", NETWORKS]
if RESTRICT:
    for spec in RESTRICT.split():
        cmd += ["--at", spec]
if PASSPHRASE: cmd += ["--passphrase", PASSPHRASE]
if SHARD: cmd += ["--shard", SHARD]

proc = subprocess.run(cmd, text=True)
print("exit code:", proc.returncode, "(0 = recovered, 1 = no match, 2 = bad input)")

## 6. Results

`targets/results.json` holds every match; the phrase is printed above as well.
Check the recovered phrase by deriving its addresses once more:

In [ ]:
import json
try:
    data = json.load(open("targets/results.json"))
    for hit in data["hits"]:
        print("network :", hit["network"])
        print("path    :", hit["path"])
        print("address :", hit["address"])
        print("phrase  :", hit["phrase"])
        print()
        subprocess.run(["target/release/spr", "derive", "--phrase", hit["phrase"],
                        "--network", hit["network"]])
except FileNotFoundError:
    print("no results file yet — run cell 5 first")

## 7. Optional: the web dashboard

Starts the dashboard and (on Colab) opens a forwarded link. Stop the cell to stop the server.

In [ ]:
import subprocess, time, os, sys

server = subprocess.Popen(["target/release/spr", "serve", "--host", "0.0.0.0", "--port", "8080"])
time.sleep(2)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8080, path="/")
except Exception:
    print("Open http://<your-instance-ip>:8080 (open port 8080 in the firewall)")
    print("Kaggle: use the 'Ports' panel or an ngrok tunnel")
try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()
    print("dashboard stopped")

## 8. Scaling tips

* More cores beat more clock: `--threads 0` uses everything available.
* `--at POS=w1,w2,w3` shrinks the space by `list_len/2048` per restricted position.
* Split over runtimes/machines with `--shard 1/4`, `--shard 2/4`, … (each shard checks
  one slice of the same space; the first one that finds a match prints it).
* `--breadth` and `--path` control how many addresses are checked per seed — keep them
  small unless you really have many funded addresses in that wallet.
* Exit code 1 with no match means "the phrase is not in the space you described":
  double-check word counts, the passphrase, and that the target address belongs to the
  network you selected.